# Co-Trip Detection & Social Riding Analysis
**NYC Citi Bike — July & August 2025**

Identifies groups of riders who appear to be traveling together ("co-trips")
by finding trips that share the same origin/destination station pair within
a tight departure and arrival window.

**Pipeline:**
1. Self-join trips on (start_station, end_station) with configurable time gaps
2. Build a co-trip graph and extract connected components → group IDs
3. Compute social riding rates by hour, weekday/weekend, and member type
4. Sensitivity analysis over threshold choices

## 0. Setup

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, count, hour, dayofweek, lit, unix_timestamp, abs as spark_abs
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load data & prepare keys

In [0]:
df = spark.table("citibike.trips")

df = (
    df
    .filter(col("start_station_name").isNotNull() & col("end_station_name").isNotNull())
    .withColumn("hour", hour("started_at"))
    .withColumn("dow", dayofweek("started_at"))
    .withColumn("is_weekend", col("dow").isin(1, 7).cast("int"))
    .withColumn("started_ts", unix_timestamp("started_at"))
    .withColumn("ended_ts", unix_timestamp("ended_at"))
)

# Assign a monotonically increasing id to uniquely reference each trip.
# If your table already has a unique trip_id column, use that instead.
df = df.withColumn("trip_id", F.monotonically_increasing_id())

print(f"Trips after filtering nulls: {df.count():,}")
df.printSchema()

## 2. Self-join to find candidate co-trip pairs

Two trips are **co-trip candidates** when they share the same OD pair and
depart/arrive within configurable windows. We only keep pairs where
`trip_a.ride_id < trip_b.ride_id` to avoid duplicates and self-joins.

In [0]:
DEP_GAP_SEC = 120   # max seconds between departures
ARR_GAP_SEC = 300   # max seconds between arrivals

a = df.alias("a")
b = df.alias("b")

pairs = (
    a.join(
        b,
        on=[
            col("a.start_station_name") == col("b.start_station_name"),
            col("a.end_station_name") == col("b.end_station_name"),
            col("a.trip_id") < col("b.trip_id"),
            spark_abs(col("a.started_ts") - col("b.started_ts")) <= DEP_GAP_SEC,
            spark_abs(col("a.ended_ts") - col("b.ended_ts")) <= ARR_GAP_SEC,
        ],
        how="inner",
    )
    .select(
        col("a.trip_id").alias("src"),
        col("b.trip_id").alias("dst"),
    )
)

print(f"Co-trip candidate pairs: {pairs.count():,}")

## 3. Connected components via Union-Find 
(no GraphFrames)

The pairs edge list is small enough to collect to the driver.
We run a standard Union-Find in pure Python to get transitive group IDs,
then broadcast the mapping back to Spark.

In [0]:
edges_pdf = pairs.toPandas()
print(f"Edges collected: {len(edges_pdf):,}")

# --- Union-Find (array-based, no iterrows) ---
src = edges_pdf["src"].values
dst = edges_pdf["dst"].values

# Map trip_ids to a dense 0..N-1 index
all_ids = np.unique(np.concatenate([src, dst]))
id_to_idx = {v: i for i, v in enumerate(all_ids)}
n = len(all_ids)

src_idx = np.array([id_to_idx[v] for v in src], dtype=np.int64)
dst_idx = np.array([id_to_idx[v] for v in dst], dtype=np.int64)

parent = np.arange(n, dtype=np.int64)
rank = np.zeros(n, dtype=np.int64)

def find(x):
    root = x
    while parent[root] != root:
        root = parent[root]
    while parent[x] != root:          # path compression
        parent[x], x = root, parent[x]
    return root

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry:
        return
    if rank[rx] < rank[ry]:
        rx, ry = ry, rx
    parent[ry] = rx
    if rank[rx] == rank[ry]:
        rank[rx] += 1

for i in range(len(src_idx)):
    union(src_idx[i], dst_idx[i])

# Vectorised root resolution
roots = np.array([find(i) for i in range(n)], dtype=np.int64)

comp_pdf = pd.DataFrame({
    "trip_id": all_ids,
    "group_id": all_ids[roots],
})
comp_pdf["group_size"] = comp_pdf.groupby("group_id")["trip_id"].transform("count")

print(f"Trips involved in a co-trip: {len(comp_pdf):,}")
print(f"Distinct groups:             {comp_pdf['group_id'].nunique():,}")
print(comp_pdf["group_size"].describe())


## 4. Broadcast group IDs back to the full trip table

In [0]:
comp_sdf = spark.createDataFrame(comp_pdf[["trip_id", "group_id", "group_size"]])

df_tagged = (
    df
    .join(comp_sdf, on="trip_id", how="left")
    .withColumn("group_size", F.coalesce(col("group_size"), lit(1)))
    .withColumn("is_cotrip", (col("group_size") > 1).cast("int"))
)

total = df_tagged.count()
cotrip_total = df_tagged.filter(col("is_cotrip") == 1).count()
print(f"Total trips: {total:,}")
print(f"Co-trips:    {cotrip_total:,}  ({100*cotrip_total/total:.2f}%)")

## 5. Group-size distribution

In [0]:
size_dist = (
    df_tagged
    .filter(col("is_cotrip") == 1)
    .select("group_id", "group_size")
    .distinct()
    .groupBy("group_size")
    .agg(count("*").alias("n_groups"))
    .orderBy("group_size")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(size_dist["group_size"].astype(str), size_dist["n_groups"], color="steelblue")
ax.set_xlabel("Group size (riders)")
ax.set_ylabel("Number of groups (log scale)")
ax.set_title("Co-Trip Group Size Distribution")
ax.set_yscale('log')
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

size_dist

## 6. Social riding rate by hour of day

In [0]:
hourly = (
    df_tagged
    .groupBy("hour")
    .agg(
        count("*").alias("total_trips"),
        F.sum("is_cotrip").alias("cotrip_trips"),
    )
    .withColumn("cotrip_rate", col("cotrip_trips") / col("total_trips"))
    .orderBy("hour")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(hourly["hour"], hourly["total_trips"], alpha=0.3, color="grey", label="Total trips")
ax1.set_xlabel("Hour of day")
ax1.set_ylabel("Total trips", color="grey")

ax2 = ax1.twinx()
ax2.plot(hourly["hour"], hourly["cotrip_rate"] * 100, "o-", color="crimson", linewidth=2, label="Co-trip rate %")
ax2.set_ylabel("Co-trip rate (%)", color="crimson")

ax1.set_title("Co-Trip Rate by Hour of Day")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
ax1.set_xticks(range(0, 24))
ax1.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 6b. Social riding rate by hour — weekday vs. weekend

In [0]:
hourly_daytype = (
    df_tagged
    .groupBy("hour", "is_weekend")
    .agg(
        count("*").alias("total_trips"),
        F.sum("is_cotrip").alias("cotrip_trips"),
    )
    .withColumn("cotrip_rate", col("cotrip_trips") / col("total_trips"))
    .orderBy("is_weekend", "hour")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
for daytype, label, color in [(0, "Weekday", "steelblue"), (1, "Weekend", "darkorange")]:
    sub = hourly_daytype[hourly_daytype["is_weekend"] == daytype].sort_values("hour")
    ax.plot(sub["hour"], sub["cotrip_rate"] * 100, "o-", color=color, linewidth=2, label=label)

ax.set_xlabel("Hour of day")
ax.set_ylabel("Co-trip rate (%)")
ax.set_title("Co-Trip Rate by Hour — Weekday vs. Weekend")
ax.set_xticks(range(0, 24))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Weekday vs. weekend

In [0]:
by_daytype = (
    df_tagged
    .groupBy("is_weekend")
    .agg(
        count("*").alias("total_trips"),
        F.sum("is_cotrip").alias("cotrip_trips"),
    )
    .withColumn("cotrip_rate", col("cotrip_trips") / col("total_trips"))
    .toPandas()
)

by_daytype["label"] = by_daytype["is_weekend"].map({0: "Weekday", 1: "Weekend"})
print(by_daytype[["label", "total_trips", "cotrip_trips", "cotrip_rate"]])

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(by_daytype["label"], by_daytype["cotrip_rate"] * 100, color=["steelblue", "darkorange"])
ax.set_ylabel("Co-trip rate (%)")
ax.set_title("Co-Trip Rate: Weekday vs. Weekend")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Member vs. casual

In [0]:
by_member = (
    df_tagged
    .groupBy("member_casual")
    .agg(
        count("*").alias("total_trips"),
        F.sum("is_cotrip").alias("cotrip_trips"),
    )
    .withColumn("cotrip_rate", col("cotrip_trips") / col("total_trips"))
    .toPandas()
)

print(by_member)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(by_member["member_casual"], by_member["cotrip_rate"] * 100, color=["teal", "coral"])
ax.set_ylabel("Co-trip rate (%)")
ax.set_title("Co-Trip Rate: Member vs. Casual")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Hour × day-type × rider-type heatmap

In [0]:
cross = (
    df_tagged
    .groupBy("hour", "is_weekend", "member_casual")
    .agg(
        count("*").alias("total"),
        F.sum("is_cotrip").alias("cotrips"),
    )
    .withColumn("rate", col("cotrips") / col("total"))
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for idx, (daytype, label) in enumerate([(0, "Weekday"), (1, "Weekend")]):
    ax = axes[idx]
    sub = cross[cross["is_weekend"] == daytype]
    for mtype, color in [("member", "steelblue"), ("casual", "coral")]:
        s = sub[sub["member_casual"] == mtype].sort_values("hour")
        ax.plot(s["hour"], s["rate"] * 100, "o-", color=color, linewidth=2, label=mtype)
    ax.set_xlabel("Hour of day")
    ax.set_title(f"Co-Trip Rate — {label}")
    ax.set_xticks(range(0, 24, 3))
    ax.legend()
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Co-trip rate (%)")
fig.suptitle("Social Riding Rate by Hour, Day Type & Rider Type", fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Sensitivity analysis over threshold choice

Re-runs the self-join for a grid of (departure gap, arrival gap) values
and reports the resulting co-trip rate. This validates that the chosen
thresholds sit on a reasonable plateau rather than an inflection point.

**Note:** this is the most expensive step — reduce `GRID` if needed.

In [0]:
GRID = [
    (60, 120),
    (60, 300),
    (120, 180),
    (120, 300),
    (120, 600),
    (180, 300),
    (180, 600),
    (300, 600),
]

results = []

for dep_gap, arr_gap in GRID:
    p = (
        a.join(
            b,
            on=[
                col("a.start_station_name") == col("b.start_station_name"),
                col("a.end_station_name") == col("b.end_station_name"),
                col("a.trip_id") < col("b.trip_id"),
                spark_abs(col("a.started_ts") - col("b.started_ts")) <= dep_gap,
                spark_abs(col("a.ended_ts") - col("b.ended_ts")) <= arr_gap,
            ],
            how="inner",
        )
    )
    # Count distinct trips involved in at least one pair (avoids full CC for speed)
    n_paired = (
        p.select(col("a.trip_id").alias("tid"))
        .union(p.select(col("b.trip_id").alias("tid")))
        .distinct()
        .count()
    )
    rate = n_paired / total
    results.append({"dep_gap_s": dep_gap, "arr_gap_s": arr_gap, "paired_trips": n_paired, "rate": rate})
    print(f"  dep≤{dep_gap:3d}s  arr≤{arr_gap:3d}s  →  {n_paired:>8,} trips  ({rate:.4%})")

sensitivity_df = pd.DataFrame(results)

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
for dep_gap, grp in sensitivity_df.groupby("dep_gap_s"):
    grp_sorted = grp.sort_values("arr_gap_s")
    ax.plot(grp_sorted["arr_gap_s"], grp_sorted["rate"] * 100, "o-", label=f"dep ≤ {dep_gap}s")

ax.set_xlabel("Arrival gap threshold (seconds)")
ax.set_ylabel("Co-trip rate (%)")
ax.set_title("Sensitivity: Co-Trip Rate vs. Threshold Choice")
ax.legend(title="Departure gap")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Save co-trip-tagged table to Delta

In [0]:
save_cols = [
    "trip_id", "started_at", "ended_at",
    "start_station_name", "end_station_name",
    "member_casual", "hour", "is_weekend",
    "group_id", "group_size", "is_cotrip",
]

(
    df_tagged
    .select(*save_cols)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable("citibike.trips_cotrip_tagged")
)
print("Saved citibike.trips_cotrip_tagged ✅")
